# **imports**

In [ ]:
import pandas as pd
import numpy as np
import datetime
import os

!pip install wget
!pip install modis-tools
!pip install pyhdf
!pip install sentinelhub
!pip install pvlib
!pip install swifter

from google.colab import drive
drive.mount('/content/drive', force_remount= True) #Use path /content/drive/MyDrive/Aerosol/files/ to access files

bbox = [34.4,-11.8,145.9,45.4]
start_date="2021-01-01"
end_date="2021-02-28"
start_date_obj = datetime.datetime.strptime(start_date, "%Y-%m-%d")
end_date_obj = datetime.datetime.strptime(end_date, "%Y-%m-%d")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.0/236.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 36.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16505 sha256=41848f24f240c72de6249b043d8a2f504ddfedd559f746ab71e6b2f96e570d62
  Stored in directory: /root/.cache/pip/wheels/ef/7f/bd/9bed48f078f3ee1fa75e0b29b6e0335ce1cb03a38d3443b3a3
Successfully built swifter
Mounted at /content/drive


# **AERONET**

### Clone github code to download aeronet data

In [ ]:
!git clone https://github.com/inciteleal/dad_download_aeronet_data.git /content/drive/MyDrive/Aerosol/bin/dad_download_aeronet_data

Cloning into '/content/drive/MyDrive/Aerosol/dad_download_aeronet_data'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 25 (delta 11), reused 20 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 36.89 KiB | 3.35 MiB/s, done.
Resolving deltas: 100% (11/11), done.


### Setting up for the code

filter all the sites that are in bbox

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Aerosol/bin/dad_download_aeronet_data/aeronet_locations_v3_og.csv", skiprows=1)

min_lon, min_lat, max_lon, max_lat = bbox
df['Latitude(decimal_degrees)'] = df['Latitude(decimal_degrees)'].astype(float)
df['Longitude(decimal_degrees)'] = df['Longitude(decimal_degrees)'].astype(float)

df_filtered = df[
    (df['Latitude(decimal_degrees)'] >= min_lat) & (df['Latitude(decimal_degrees)'] <= max_lat) &
    (df['Longitude(decimal_degrees)'] >= min_lon) & (df['Longitude(decimal_degrees)'] <= max_lon)
]

df_filtered.to_csv("/content/drive/MyDrive/Aerosol/bin/dad_download_aeronet_data/aeronet_locations_v3.csv", index=False)

create input.csv to be used by dad.py

In [ ]:
locations_df = pd.read_csv("/content/drive/MyDrive/Aerosol/bin/dad_download_aeronet_data/aeronet_locations_v3.csv")
site_names = locations_df['Site_Name'].dropna().unique()

input_data = {
    "year_initial": start_date_obj.year,
    "month_initial": start_date_obj.month,
    "day_initial": start_date_obj.day,
    "year_final": end_date_obj.year,
    "month_final": end_date_obj.month,
    "day_final": end_date_obj.day,
    "level": 15,
    "avg": 10,
    "download": "on"
}

input_rows = []
products = ["lid"]

for site in site_names:
    for product in products:
        row = input_data.copy()
        row["site"] = site
        row["products"] = product
        input_rows.append(row)

input_df = pd.DataFrame(input_rows)

input_df = input_df[[
    "year_initial", "month_initial", "day_initial",
    "year_final", "month_final", "day_final",
    "site", "level", "avg", "products", "download"
]]

output_path = "/content/drive/MyDrive/Aerosol/dad_download_aeronet_data/01-input_dir/input.csv"
input_df.to_csv(output_path, index=False, quotechar='"')

print(f"input.csv created with {len(input_df)} rows (separate rows for each product).")

input.csv created with 428 rows (separate rows for each product).


clear all previous files that are not needed anymore

In [ ]:
!rm /content/drive/MyDrive/Aerosol/dad_download_aeronet_data/01-rawdata/*

In [ ]:
drive.mount('/content/drive', force_remount= True)

Mounted at /content/drive


In [ ]:
# @title
!python /content/drive/MyDrive/Aerosol/dad_download_aeronet_data/dad.py
drive.mount('/content/drive', force_remount= True)

['input.csv']

Downloaded lid20210101_20210228_Ussuriysk_level15.csv

Downloaded lid20210101_20210228_Hong_Kong_Hok_Tsui_level15.csv

Downloaded lid20210101_20210228_SEDE_BOKER_level15.csv

Downloaded lid20210101_20210228_MALE_level15.csv

Downloaded lid20210101_20210228_Dalanzadgad_level15.csv

Downloaded lid20210101_20210228_Kaashidhoo_level15.csv

Downloaded lid20210101_20210228_NCU_Taiwan_level15.csv

Downloaded lid20210101_20210228_Dongsha_Island_level15.csv

Downloaded lid20210101_20210228_Chao_Jou_level15.csv

Downloaded lid20210101_20210228_MCO-Hanimaadhoo_level15.csv

Downloaded lid20210101_20210228_Bahrain_level15.csv

Downloaded lid20210101_20210228_Solar_Village_level15.csv

Downloaded lid20210101_20210228_Chinhae_level15.csv

Downloaded lid20210101_20210228_GOA_INDIA_level15.csv

Downloaded lid20210101_20210228_Dharwar_level15.csv

Downloaded lid20210101_20210228_Anmyon_level15.csv

Downloaded lid20210101_20210228_Dead_Sea_level15.csv

Downloaded lid20210101_20210228_Nes_Z

output hidden beacuse its too long

In [ ]:
def read_corrected_csv(filepath):
    df = pd.read_csv(filepath, skiprows=6)
    df = df.dropna(axis=1, how='all')
    df['source_file'] = os.path.basename(filepath)
    return df

def combine_files(folder_path, prefix):
    errors=0
    success=0
    combined_df = pd.DataFrame()

    for filename in os.listdir(folder_path):
        if filename.startswith(prefix) and filename.endswith('.csv'):
            filepath = os.path.join(folder_path, filename)
            try:
                df = read_corrected_csv(filepath)
                combined_df = pd.concat([combined_df, df], ignore_index=True)
                success+=1
                print(f"Successfully combined {filename}.")
            except Exception as e:
                errors+=1
    print(f"Total errors: {errors}")
    print(f"Total success: {success}")
    return combined_df

input_folder = '/content/drive/MyDrive/Aerosol/dad_download_aeronet_data/01-rawdata/'

lid_combined = combine_files(input_folder, 'lid')
lid_combined.to_csv('/content/drive/MyDrive/Aerosol/dad_download_aeronet_data/lid.csv', index=False)
print("Saved combined LID CSV.")


Successfully combined lid20210101_20210228_Ussuriysk_level15.csv.
Successfully combined lid20210101_20210228_SEDE_BOKER_level15.csv.
Successfully combined lid20210101_20210228_Dalanzadgad_level15.csv.
Successfully combined lid20210101_20210228_Dongsha_Island_level15.csv.
Successfully combined lid20210101_20210228_Anmyon_level15.csv.
Successfully combined lid20210101_20210228_Osaka_level15.csv.
Successfully combined lid20210101_20210228_Shirahama_level15.csv.
Successfully combined lid20210101_20210228_Kanpur_level15.csv.
Successfully combined lid20210101_20210228_Beijing_level15.csv.
Successfully combined lid20210101_20210228_Seoul_SNU_level15.csv.
Successfully combined lid20210101_20210228_XiangHe_level15.csv.
Successfully combined lid20210101_20210228_Noto_level15.csv.
Successfully combined lid20210101_20210228_Kuwait_University_level15.csv.
Successfully combined lid20210101_20210228_Gwangju_GIST_level15.csv.
Successfully combined lid20210101_20210228_Mezaira_level15.csv.
Successfully

output hidden beacuse its too long

# **MODIS**

In [ ]:
from modis_tools.auth import ModisSession
from modis_tools.resources import CollectionApi, GranuleApi
from modis_tools.granule_handler import GranuleHandler

In [ ]:
session = ModisSession(username=os.getenv("ModisUsername"), password=os.getenv("ModisPassword"))

In [ ]:
drive.mount('/content/drive', force_remount= True)

Mounted at /content/drive


### MCD12C1 (L3) data download

In [ ]:
collection_client = CollectionApi(session=session)
collections = collection_client.query(short_name="MCD12C1", version="061")
if collections:
    print(collections)
    granule_client = GranuleApi.from_collection(collections[0], session=session)

    granules = granule_client.query(start_date=start_date, end_date=end_date, bounding_box=bbox)

    os.makedirs("/content/drive/MyDrive/Aerosol/files/MCD12C1_L3", exist_ok=True)
    GranuleHandler.download_from_granules(granules, session, path="/content/drive/MyDrive/Aerosol/files/MCD12C1_L3")
else:
    print("No collections found matching the query.")

[Collection(id='C2484078896-LPCLOUD', title='MODIS/Terra+Aqua Land Cover Type Yearly L3 Global 0.05Deg CMG V061', dataset_id='MODIS/Terra+Aqua Land Cover Type Yearly L3 Global 0.05Deg CMG V061', coordinate_system='CARTESIAN', time_start='2001-01-01T00:00:00.000Z', updated=datetime.datetime(2022, 8, 5, 0, 0, tzinfo=datetime.timezone.utc), links=[CollectionLink(hreflang='en-US', href=AnyUrl('https://e4ftl01.cr.usgs.gov/MOTA/MCD12C1.061/', scheme='https', host='e4ftl01.cr.usgs.gov', tld='gov', host_type='domain', path='/MOTA/MCD12C1.061/'), type=None), CollectionLink(hreflang='en-US', href=AnyUrl('https://search.earthdata.nasa.gov/search?q=C2484078896-LPCLOUD', scheme='https', host='search.earthdata.nasa.gov', tld='gov', host_type='domain', path='/search', query='q=C2484078896-LPCLOUD'), type=None), CollectionLink(hreflang='en-US', href=AnyUrl('https://doi.org/10.5067/MODIS/MCD12C1.061', scheme='https', host='doi.org', tld='org', host_type='domain', path='/10.5067/MODIS/MCD12C1.061'), typ

Downloading: 100%|██████████| 1/1 [00:24<00:00, 24.20s/file]


In [ ]:
import os
import numpy as np
import pandas as pd
from osgeo import gdal
from tqdm import tqdm

input_path = "/content/drive/MyDrive/Aerosol/files/MCD12C1_L3/"
clean_path = "/content/drive/MyDrive/Aerosol/files/MCD12C1_L3_cleaned/"
os.makedirs(clean_path, exist_ok=True)

SDS_LC = "Majority_Land_Cover_Type_1"
SDS_PCT = "Land_Cover_Type_1_Percent"

def get_lon_lat_grid(gt, shape):
    originX, pixelW, _, originY, _, pixelH = gt
    cols, rows = shape[1], shape[0]
    xs = originX + (np.arange(cols) + 0.5) * pixelW
    ys = originY + (np.arange(rows) + 0.5) * pixelH
    return np.meshgrid(xs, ys)

def process_file(fname):
    if not fname.lower().endswith(".hdf"):
        return

    hdf_path = os.path.join(input_path, fname)
    ds = gdal.Open(hdf_path, gdal.GA_ReadOnly)
    subds = ds.GetSubDatasets()

    sds_map = {desc: uri for uri, desc in subds}
    lc_uri = next((uri for desc, uri in sds_map.items() if SDS_LC in desc), None)
    pct_uri = next((uri for desc, uri in sds_map.items() if SDS_PCT in desc), None)

    if lc_uri is None or pct_uri is None:
        tqdm.write(f"Skipping {fname} — SDS not found.")
        return

    lc_arr = gdal.Open(lc_uri).ReadAsArray()

    # Handle multi-band pct_arr properly
    pct_dataset = gdal.Open(pct_uri)
    if pct_dataset.RasterCount > 1:
        pct_arr = pct_dataset.GetRasterBand(1).ReadAsArray()
    else:
        pct_arr = pct_dataset.ReadAsArray()

    # Ensure shapes match
    if lc_arr.shape != pct_arr.shape:
        tqdm.write(f"Skipping {fname} — Shape mismatch between LC and PCT.")
        return

    gt = gdal.Open(lc_uri).GetGeoTransform()
    lon_grid, lat_grid = get_lon_lat_grid(gt, lc_arr.shape)

    df = pd.DataFrame({
        "Longitude": lon_grid.flatten(),
        "Latitude": lat_grid.flatten(),
        "LC_IGBP": lc_arr.flatten(),
        "Pct_Agreement": pct_arr.flatten(),
        # Date as 9-12th characters in file name
        "Year": int(fname[9:13])
    })

    out_csv = os.path.join(clean_path, fname.replace(".hdf", ".csv"))
    df.to_csv(out_csv, index=False)

files = sorted(os.listdir(input_path))
for file in tqdm(files, desc="Processing"):
    process_file(file)

Processing: 100%|██████████| 1/1 [02:18<00:00, 138.27s/it]


## MYD04 (L2) data download

In [ ]:
collection_client = CollectionApi(session=session)

collections = collection_client.query(short_name="MYD04_L2")

if collections:
    print(collections)
    granule_client = GranuleApi.from_collection(collections[0], session=session)

    granules = granule_client.query(start_date=start_date, end_date=end_date, bounding_box=bbox)

    os.makedirs("/content/drive/MyDrive/Aerosol/files/MYD04_L2", exist_ok=True)
    GranuleHandler.download_from_granules(granules, session, path="/content/drive/MyDrive/Aerosol/files/MYD04_L2")
else:
    print("No collections found matching the query.")


[Collection(id='C1443533683-LAADS', title='MODIS/Aqua Aerosol 5-Min L2 Swath 10km', dataset_id='MODIS/Aqua Aerosol 5-Min L2 Swath 10km', coordinate_system='CARTESIAN', time_start='2002-07-04T00:45:00.000Z', updated=None, links=[CollectionLink(hreflang='en-US', href=AnyUrl('https://doi.org/10.5067/MODIS/MYD04_L2.061', scheme='https', host='doi.org', tld='org', host_type='domain', path='/10.5067/MODIS/MYD04_L2.061'), type=None), CollectionLink(hreflang='en-US', href=AnyUrl('https://modis-atmos.gsfc.nasa.gov/sites/default/files/ModAtmo/modis_deep_blue_c61_changes2.pdf', scheme='https', host='modis-atmos.gsfc.nasa.gov', tld='gov', host_type='domain', path='/sites/default/files/ModAtmo/modis_deep_blue_c61_changes2.pdf'), type=None), CollectionLink(hreflang='en-US', href=AnyUrl('https://modis-atmos.gsfc.nasa.gov/sites/default/files/ModAtmo/C061_Aerosol_Dark_Target_v2.pdf', scheme='https', host='modis-atmos.gsfc.nasa.gov', tld='gov', host_type='domain', path='/sites/default/files/ModAtmo/C061

Downloading: 100%|██████████| 1442/1442 [12:57<00:00,  1.85file/s]


## MYD04 conversion and cleaning

In [ ]:
from sklearn.neighbors import KDTree
from tqdm import tqdm
from collections import defaultdict
from glob import glob
from pyhdf.SD import SD, SDC
from datetime import datetime, timedelta

In [ ]:
input_path  = "/content/drive/MyDrive/Aerosol/files/MYD04_L2/"
clean_path  = "/content/drive/MyDrive/Aerosol/files/MYD04_L2_csv/"
os.makedirs(clean_path, exist_ok=True)

min_lon, min_lat, max_lon, max_lat = bbox

AOD_SDS      = "Optical_Depth_Land_And_Ocean"
ANG_SDS      = "Deep_Blue_Angstrom_Exponent_Land"
SPEC_REF_SDS = "Deep_Blue_Spectral_TOA_Reflectance_Land"

for fname in tqdm(os.listdir(input_path), desc="Processing HDF files"):
    if not fname.lower().endswith(".hdf"):
        continue

    hdf_file = os.path.join(input_path, fname)

    try:
        parts = fname.split(".")
        date_part = parts[1]
        time_part = parts[2]

        year = int(date_part[1:5])
        doy = int(date_part[5:8])
        hour = int(time_part[:2])
        minute = int(time_part[2:])

        file_datetime = datetime(year, 1, 1) + timedelta(days=doy - 1, hours=hour, minutes=minute)

        hdf = SD(hdf_file, SDC.READ)

        lat = hdf.select("Latitude")[:].flatten()
        lon = hdf.select("Longitude")[:].flatten()

        m = (
            (lon >= min_lon) & (lon <= max_lon) &
            (lat >= min_lat) & (lat <= max_lat)
        )

        aod = hdf.select(AOD_SDS)[:].flatten()[m]
        ang = hdf.select(ANG_SDS)[:].flatten()[m]

        spec = hdf.select(SPEC_REF_SDS)[:]
        bands = spec.shape[0]

        if bands >= 6:
            idx_412, idx_470, idx_660 = 0, 2, 5
        elif bands == 3:
            idx_412, idx_470, idx_660 = 0, 1, 2
        else:
            raise ValueError(f"Unexpected {bands} bands in {SPEC_REF_SDS}")

        r412 = spec[idx_412].flatten()[m]
        r470 = spec[idx_470].flatten()[m]
        r660 = spec[idx_660].flatten()[m]

        hdf.end()

        df = pd.DataFrame({
            "Date":              file_datetime.date().isoformat(),
            "Time":              file_datetime.time().isoformat(),
            "Latitude":          lat[m],
            "Longitude":         lon[m],
            "AOD_550nm":         aod,
            "Angstrom_Exponent": ang,
            "Reflect_412nm":     r412,
            "Reflect_470nm":     r470,
            "Reflect_660nm":     r660,
        })

        out_csv = os.path.join(clean_path, fname.replace(".hdf", ".csv"))
        df.to_csv(out_csv, index=False)

    except Exception as e:
        print(f"⛔ Failed {fname}: {e}")
        continue

print(" All done!")


Processing HDF files: 100%|██████████| 1442/1442 [04:35<00:00,  5.24it/s]

 All done!


In [ ]:
clean_path = "/content/drive/MyDrive/Aerosol/files/MYD04_L2_csv/"
output_path = "/content/drive/MyDrive/Aerosol/files/MYD04_L2_merged_bydate/"
os.makedirs(output_path, exist_ok=True)

date_groups = defaultdict(list)

for file_path in glob(os.path.join(clean_path, "*.csv")):
    try:
        df = pd.read_csv(file_path, parse_dates=["Date"])
        for date_val, group in df.groupby("Date"):
            date_str = date_val.strftime("%Y%m%d")
            date_groups[date_str].append(group)
    except Exception as e:
        print(f"⛔ Error reading {file_path}: {e}")
        continue

for date_str, dfs in date_groups.items():
    combined = pd.concat(dfs, ignore_index=True)

    merged_filename = f"MYD04_L2_{date_str}_merged.csv"
    combined.to_csv(os.path.join(output_path, merged_filename), index=False)
    print(f"Merged and saved for date {date_str} → {merged_filename}")


Merged and saved for date 20210227 → MYD04_L2_20210227_merged.csv
Merged and saved for date 20210226 → MYD04_L2_20210226_merged.csv
Merged and saved for date 20210225 → MYD04_L2_20210225_merged.csv
Merged and saved for date 20210224 → MYD04_L2_20210224_merged.csv
Merged and saved for date 20210223 → MYD04_L2_20210223_merged.csv
Merged and saved for date 20210222 → MYD04_L2_20210222_merged.csv
Merged and saved for date 20210221 → MYD04_L2_20210221_merged.csv
Merged and saved for date 20210220 → MYD04_L2_20210220_merged.csv
Merged and saved for date 20210219 → MYD04_L2_20210219_merged.csv
Merged and saved for date 20210218 → MYD04_L2_20210218_merged.csv
Merged and saved for date 20210217 → MYD04_L2_20210217_merged.csv
Merged and saved for date 20210216 → MYD04_L2_20210216_merged.csv
Merged and saved for date 20210215 → MYD04_L2_20210215_merged.csv
Merged and saved for date 20210214 → MYD04_L2_20210214_merged.csv
Merged and saved for date 20210213 → MYD04_L2_20210213_merged.csv
Merged and

output hidden beacuse its too long

In [ ]:
input_path = "/content/drive/MyDrive/Aerosol/files/MYD04_L2_merged_bydate/"

all_files = sorted(glob(os.path.join(input_path, "*.csv")))

all_dfs = []
for file in all_files:
    try:
        df = pd.read_csv(file, parse_dates=["Date"])
        all_dfs.append(df)
    except Exception as e:
        print(f"⛔ Error reading {file}: {e}")

final_df = pd.concat(all_dfs, ignore_index=True)
print(f"Total rows: {len(final_df)}")
columns_to_check = ['AOD_550nm', 'Angstrom_Exponent', 'Reflect_412nm', 'Reflect_470nm', 'Reflect_660nm']
modis_cleaned = final_df[~final_df[columns_to_check].isin([-9999]).any(axis=1)]
print(f"Rows after filtering: {len(modis_cleaned)}")
modis_cleaned.to_csv("/content/drive/MyDrive/Aerosol/files/MYD04.csv", index=False)
print(f"All files merged into: /content/drive/MyDrive/Aerosol/files/MYD04.csv")


Total rows: 22804722
Rows after filtering: 1347022
All files merged into: /content/drive/MyDrive/Aerosol/files/MYD04.csv


**2 crore rows -> 13 lakh rows, too much missing data (that too we have not filtered out ocean)**

# **TROPOMI**

In [ ]:
from sentinelhub import (
    SentinelHubCatalog, SentinelHubRequest, SHConfig, BBox, CRS, MimeType,
    DataCollection, bbox_to_dimensions
)

In [ ]:
config = SHConfig()
config.sh_client_id = os.getenv("CLIENT_ID")
config.sh_client_secret = os.getenv("CLIENT_SECRET")
config.sh_base_url = 'https://sh.dataspace.copernicus.eu'
config.sh_token_url = (
    'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/'
    'protocol/openid-connect/token'
)

### TROPOMI Sentinel 5P data download using Sentinel Hub

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Load Natural Earth land polygons for land mask
world = gpd.read_file('/content/drive/MyDrive/Aerosol/ne_110m_land/ne_110m_land.shp')
land = world.union_all()  # Single MultiPolygon of all land masses

def is_land(lat, lon):
    return land.contains(Point(lon, lat))

resolution = 7000

time_interval = (start_date, end_date)

data_collection = DataCollection.SENTINEL5P.define_from("s5p", service_url=config.sh_base_url)

all_tiles = []
lon_steps = range(int(bbox[0]), int(bbox[2]), 30)
lat_steps = range(int(bbox[1]), int(bbox[3]), 30)

for lon_min in lon_steps:
    for lat_min in lat_steps:
        lon_max = lon_min + 30
        lat_max = lat_min + 30

        print(f"\n🌍 Processing tile: ({lon_min}, {lat_min}) to ({lon_max}, {lat_max})")

        bboxt = BBox((lon_min, lat_min, lon_max, lat_max), crs=CRS.WGS84)
        bbox_size = bbox_to_dimensions(bboxt, resolution=resolution)
        bbox_size = (
            max(1, min(bbox_size[0], 2500)),
            max(1, min(bbox_size[1], 2500))
        )

        catalog = SentinelHubCatalog(config=config)
        search_results = catalog.search(
            collection=data_collection,
            bbox=bboxt,
            time=time_interval,
            fields={"include": ["id", "properties.datetime"], "exclude": []}
        )

        search_results = list(search_results)
        COtiles = [tile for tile in search_results if "RPRO_L2__CO" in tile['id']]
        AItiles = [tile for tile in search_results if "RPRO_L2__AER_AI" in tile['id']]
        NO2tiles = [tile for tile in search_results if "RPRO_L2__NO2" in tile['id']]

        bands = {
            'CO': ('CO', COtiles),
            'AER_AI': ('AER_AI_354_388', AItiles),
            'NO2': ('NO2', NO2tiles)
        }

        df_band = {}

        for band_name, band_info in bands.items():
            print(f"\n Searching band: {band_name}\n")
            evalscript = f"""
//VERSION=3
function setup() {{
  return {{ input: [{{ bands: [\"{band_info[0]}\"] }}], output: {{ bands: 1, sampleType: \"FLOAT32\" }} }};
}}
function evaluatePixel(sample) {{
  return [sample.{band_info[0]}];
}}
"""

            band_dfs = []
            for tile in band_info[1]:
                ts = tile['properties']['datetime']
                print(f"{band_name} tile at {pd.to_datetime(ts)} - {tile['id']}")

                request = SentinelHubRequest(
                    evalscript=evalscript,
                    input_data=[SentinelHubRequest.input_data(
                        data_collection=data_collection,
                        time_interval=(ts, ts)
                    )],
                    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
                    bbox=bboxt,
                    size=bbox_size,
                    config=config
                )
                data = request.get_data()[0]
                h, w = data.shape

                min_lon, min_lat = bboxt.lower_left
                max_lon, max_lat = bboxt.upper_right
                lons = np.linspace(min_lon, max_lon, w)
                lats = np.linspace(max_lat, min_lat, h)
                lon_grid, lat_grid = np.meshgrid(lons, lats)

                flat_vals = data.flatten()
                valid = ~np.isnan(flat_vals)

                ts_dt = pd.to_datetime(ts)
                df = pd.DataFrame({
                    'latitude': lat_grid.flatten()[valid],
                    'longitude': lon_grid.flatten()[valid],
                    band_name: flat_vals[valid],
                    'datetime': ts_dt
                })
                if df.empty:
                    print("DataFrame is empty, skipping land filtering and column drop.")
                    continue
                df['on_land'] = df.apply(lambda row: is_land(row['latitude'], row['longitude']), axis=1)
                df = df[df['on_land']].drop(columns=['on_land'])

                band_dfs.append(df)

            if band_dfs:
                df_band[band_name] = pd.concat(band_dfs, ignore_index=True)
            else:
                df_band[band_name] = pd.DataFrame()

        if not df_band['CO'].empty:
            merged = df_band['CO']
            for band_name in ['AER_AI', 'NO2']:
                merged = pd.merge(
                    merged,
                    df_band[band_name],
                    on=['latitude', 'longitude', 'datetime'],
                    how='inner'
                )

            merged['year'] = merged['datetime'].dt.year
            merged['month'] = merged['datetime'].dt.month
            merged['day'] = merged['datetime'].dt.day
            merged['time'] = merged['datetime'].dt.strftime('%H:%M:%S')
            merged.drop(columns=['datetime'], inplace=True)

            all_tiles.append(merged)

if all_tiles:
    final_df = pd.concat(all_tiles, ignore_index=True)
    final_df.to_csv("/content/drive/MyDrive/Aerosol/files/Tropomi.csv", index=False)
    print("✅ Saved Tropomi.csv with global data")
else:
    print("⚠️ No data collected.")



🌍 Processing tile: (34, -11) to (64, 19)

 Searching band: CO

CO tile at 2021-01-15 10:36:45+00:00 - S5P_RPRO_L2__CO_____20210115T103645_20210115T121815_16884_03_020400_20220905T211604.nc
CO tile at 2021-01-15 08:55:15+00:00 - S5P_RPRO_L2__CO_____20210115T085515_20210115T103645_16883_03_020400_20220905T211603.nc
CO tile at 2021-01-15 07:13:45+00:00 - S5P_RPRO_L2__CO_____20210115T071345_20210115T085515_16882_03_020400_20220905T211603.nc
DataFrame is empty, skipping land filtering and column drop.
CO tile at 2021-01-14 10:55:43+00:00 - S5P_RPRO_L2__CO_____20210114T105543_20210114T123714_16870_03_020400_20220905T205909.nc
CO tile at 2021-01-14 09:14:13+00:00 - S5P_RPRO_L2__CO_____20210114T091413_20210114T105543_16869_03_020400_20220905T205908.nc
CO tile at 2021-01-14 07:32:43+00:00 - S5P_RPRO_L2__CO_____20210114T073243_20210114T091413_16868_03_020400_20220905T205908.nc
CO tile at 2021-01-13 11:14:42+00:00 - S5P_RPRO_L2__CO_____20210113T111442_20210113T125612_16856_03_020400_20220905T205

output hidden beacuse its too long

In [ ]:
final_df.shape

(5192519, 9)

In [ ]:
from pvlib.solarposition import get_solarposition

tropomi = pd.read_csv('/content/drive/MyDrive/Aerosol/Aaditya/Tropomi.csv')

tropomi['datetime'] = pd.to_datetime(
    tropomi[['year', 'month', 'day']].astype(str).agg('-'.join, axis=1) + ' ' + tropomi['time'],
    utc=True
)

print("⚙️ Calculating SZA in vectorized mode...")
solpos = get_solarposition(
    time=tropomi['datetime'],
    latitude=tropomi['latitude'].values,
    longitude=tropomi['longitude'].values,
    altitude=0,
    pressure=None,
    temperature=12
)

tropomi['SZA'] = solpos['zenith'].values

tropomi.to_csv('/content/drive/MyDrive/Aerosol/Aaditya/Tropomi-2021-01-16-2021-01-31.csv', index=False)
print("✅ Done! Saved Tropomi_with_SZA.csv")